# Agentic semantic policy–sentiment gap analysis

This pipeline is independent of the two causal-NLP analyses. It reads only the
shared cleaned sentence inventory:

```text
causal_nlp/output/shared/clean_sentence_inventory.csv
```

It performs two DeepSeek passes for each evidence package:

1. evidence-grounded semantic gap analysis;
2. independent verification using reordered evidence.

Machine-retained findings require repeated-run agreement, valid evidence from both
corpora, sufficient confidence, and sufficient evidence faithfulness. Human labels
are preserved when the notebook is rerun.


In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
import random
import re
import sys
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# Reproducibility and workload
RANDOM_STATE = int(os.environ.get("AGENT_RANDOM_STATE", "42"))
ANALYSIS_RUNS = max(1, int(os.environ.get("AGENT_ANALYSIS_RUNS", "2")))
BATCHES_PER_SCOPE = max(1, int(os.environ.get("AGENT_BATCHES_PER_SCOPE", "2")))
SENTENCES_PER_CORPUS = max(4, int(os.environ.get("AGENT_SENTENCES_PER_CORPUS", "18")))
MAX_FINDINGS = max(1, int(os.environ.get("AGENT_MAX_FINDINGS", "5")))
MAX_CASES = max(0, int(os.environ.get("AGENT_MAX_CASES", "0")))

# Scope eligibility and retention criteria
MIN_SCOPE_SENTENCES = max(1, int(os.environ.get("AGENT_MIN_SCOPE_SENTENCES", "12")))
MIN_SCOPE_SOURCES = max(1, int(os.environ.get("AGENT_MIN_SCOPE_SOURCES", "2")))
MIN_CONFIDENCE = float(os.environ.get("AGENT_MIN_CONFIDENCE", "0.60"))
MIN_FAITHFULNESS = float(os.environ.get("AGENT_MIN_FAITHFULNESS", "0.80"))
MIN_STABILITY = float(os.environ.get("AGENT_MIN_STABILITY", "0.50"))
REQUIRED_ACCEPTED_RUNS = int(
    os.environ.get("AGENT_REQUIRED_ACCEPTED_RUNS", str(ANALYSIS_RUNS))
)
REQUIRED_ACCEPTED_RUNS = min(max(1, REQUIRED_ACCEPTED_RUNS), ANALYSIS_RUNS)

RUN_AGENT = os.environ.get(
    "RUN_DEEPSEEK_AGENT",
    "1" if os.environ.get("DEEPSEEK_API_KEY") else "0",
).strip().lower() not in {"0", "false", "no", "off"}

RESUME_RUNS = os.environ.get("AGENT_RESUME_RUNS", "1").strip().lower() not in {
    "0", "false", "no", "off"
}


def find_causal_nlp_root() -> Path:
    configured = os.environ.get("CAUSAL_NLP_ROOT")
    candidates: list[Path] = []

    if configured:
        candidates.append(Path(configured).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        *cwd.parents,
        cwd / "progress" / "causal_nlp",
    ])

    checked: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in checked:
            continue
        checked.add(candidate)

        if (
            (candidate / "agent.py").exists()
            and (candidate / "output" / "shared" / "clean_sentence_inventory.csv").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate causal_nlp/agent.py and "
        "causal_nlp/output/shared/clean_sentence_inventory.csv. "
        "Set CAUSAL_NLP_ROOT to the causal_nlp directory."
    )


METHOD_ROOT = find_causal_nlp_root()
SHARED_INPUT = METHOD_ROOT / "output" / "shared" / "clean_sentence_inventory.csv"
OUTPUT_DIR = METHOD_ROOT / "output" / "agentic_semantic_gap"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(METHOD_ROOT) not in sys.path:
    sys.path.insert(0, str(METHOD_ROOT))

from agent import ask_agent

RUNS_PATH = OUTPUT_DIR / "semantic_agent_runs.jsonl"
STABILITY_PATH = OUTPUT_DIR / "semantic_agent_stability.csv"
CANDIDATES_PATH = OUTPUT_DIR / "semantic_agent_candidates.csv"
FINDINGS_PATH = OUTPUT_DIR / "semantic_agent_findings.csv"
REVIEW_PATH = OUTPUT_DIR / "semantic_agent_human_review.csv"
EVALUATION_PATH = OUTPUT_DIR / "semantic_agent_evaluation.csv"

print("Causal-NLP folder:", METHOD_ROOT)
print("Shared input:", SHARED_INPUT)
print("Output folder:", OUTPUT_DIR)
print("DeepSeek enabled:", RUN_AGENT)
print("Analysis runs:", ANALYSIS_RUNS)
print("Required accepted runs:", REQUIRED_ACCEPTED_RUNS)
print("Resume completed runs:", RESUME_RUNS)

In [ ]:
REQUIRED_COLUMNS = {
    "sentence_id",
    "clean_sentence",
    "corpus",
    "source_type",
    "source_file",
    "doc_id",
    "chunk_id",
    "country",
    "heading_context",
    "synthetic_type",
}

inventory = pd.read_csv(SHARED_INPUT)
missing = REQUIRED_COLUMNS.difference(inventory.columns)
if missing:
    raise ValueError(f"Shared inventory is missing columns: {sorted(missing)}")

for column in REQUIRED_COLUMNS:
    inventory[column] = inventory[column].fillna("").astype(str)

if inventory["sentence_id"].duplicated().any():
    duplicates = inventory.loc[
        inventory["sentence_id"].duplicated(keep=False), "sentence_id"
    ].head(10).tolist()
    raise ValueError(f"sentence_id must be unique. Examples: {duplicates}")

synthetic_mask = (
    inventory["source_type"].str.strip().str.lower().eq("synthetic")
    | inventory["synthetic_type"].str.strip().ne("")
)

empirical = inventory[
    ~synthetic_mask
    & inventory["corpus"].str.strip().str.lower().isin(["policy", "sentiment"])
].copy()

empirical["corpus"] = empirical["corpus"].str.strip().str.lower()
empirical["clean_sentence"] = empirical["clean_sentence"].str.strip()
empirical = empirical[empirical["clean_sentence"].ne("")].copy()


def infer_country(row: pd.Series) -> str:
    current = str(row.get("country", "")).strip().lower()
    if current and current not in {"nan", "none", "other", "unknown"}:
        return current.replace(" ", "_")

    source = " ".join(
        str(row.get(column, ""))
        for column in ["source_file", "doc_id", "heading_context"]
    ).lower()

    country_tokens = {
        "ireland": ["ireland", "irish", "qqi"],
        "france": ["france", "french", "français", "francais", "ifop"],
        "australia": ["australia", "australian"],
        "united_states": ["united states", "usa", "u.s.", "american"],
    }
    for country, tokens in country_tokens.items():
        if any(token in source for token in tokens):
            return country
    return "other"


empirical["analysis_country"] = empirical.apply(infer_country, axis=1)

if empirical.empty:
    raise ValueError("No empirical policy or sentiment sentences were found.")

support = empirical.groupby("corpus", as_index=False).agg(
    sentences=("sentence_id", "count"),
    sources=("source_file", "nunique"),
)
display(support)

country_support = empirical.groupby(
    ["analysis_country", "corpus"], as_index=False
).agg(
    sentences=("sentence_id", "count"),
    sources=("source_file", "nunique"),
)
display(country_support)

In [ ]:
ALLOWED_CLASSIFICATIONS = {
    "policy_gap",
    "sentiment_gap",
    "partial_alignment",
    "alignment",
    "insufficient_evidence",
}
SUBSTANTIVE_CLASSIFICATIONS = {
    "policy_gap",
    "sentiment_gap",
    "partial_alignment",
}


def evidence_item(row: pd.Series) -> dict[str, str]:
    return {
        "evidence_id": str(row["sentence_id"]),
        "corpus": str(row["corpus"]),
        "text": str(row["clean_sentence"]),
        "source_file": str(row["source_file"]),
        "doc_id": str(row["doc_id"]),
        "chunk_id": str(row["chunk_id"]),
        "country": str(row["analysis_country"]),
        "heading_context": str(row["heading_context"]),
    }


def source_balanced_sample(
    frame: pd.DataFrame,
    limit: int,
    seed: int,
) -> list[dict[str, str]]:
    """Round-robin sample across source files without replacement."""
    if frame.empty or limit <= 0:
        return []

    rng = random.Random(seed)
    grouped: dict[str, list[dict[str, Any]]] = {}

    for source, group in frame.groupby("source_file", sort=True):
        rows = group.sort_values("sentence_id").to_dict("records")
        rng.shuffle(rows)
        grouped[str(source)] = rows

    sources = list(grouped)
    rng.shuffle(sources)
    selected: list[dict[str, str]] = []

    while sources and len(selected) < limit:
        remaining_sources: list[str] = []
        for source in sources:
            rows = grouped[source]
            if rows and len(selected) < limit:
                selected.append(evidence_item(pd.Series(rows.pop())))
            if rows:
                remaining_sources.append(source)
        sources = remaining_sources

    return selected


def build_scopes(frame: pd.DataFrame) -> list[tuple[str, pd.DataFrame]]:
    scopes: list[tuple[str, pd.DataFrame]] = [("global", frame)]

    for country in sorted(frame["analysis_country"].unique()):
        if country == "other":
            continue

        subset = frame[frame["analysis_country"].eq(country)].copy()
        counts = subset.groupby("corpus")["sentence_id"].count().to_dict()
        sources = subset.groupby("corpus")["source_file"].nunique().to_dict()

        enough_sentences = all(
            counts.get(corpus, 0) >= MIN_SCOPE_SENTENCES
            for corpus in ["policy", "sentiment"]
        )
        enough_sources = all(
            sources.get(corpus, 0) >= MIN_SCOPE_SOURCES
            for corpus in ["policy", "sentiment"]
        )

        if enough_sentences and enough_sources:
            scopes.append((country, subset))

    return scopes


def build_packages(frame: pd.DataFrame) -> list[dict[str, Any]]:
    packages: list[dict[str, Any]] = []

    for scope_index, (scope, subset) in enumerate(build_scopes(frame)):
        for batch_index in range(BATCHES_PER_SCOPE):
            seed = RANDOM_STATE + scope_index * 1000 + batch_index * 100

            policy = source_balanced_sample(
                subset[subset["corpus"].eq("policy")],
                SENTENCES_PER_CORPUS,
                seed,
            )
            sentiment = source_balanced_sample(
                subset[subset["corpus"].eq("sentiment")],
                SENTENCES_PER_CORPUS,
                seed + 1,
            )

            if not policy or not sentiment:
                continue

            evidence = policy + sentiment
            random.Random(seed + 2).shuffle(evidence)

            packages.append({
                "analysis_id": f"{scope}__batch_{batch_index + 1:02d}",
                "scope": scope,
                "batch_id": batch_index + 1,
                "maximum_findings": MAX_FINDINGS,
                "evidence_counts": {
                    "policy": len(policy),
                    "sentiment": len(sentiment),
                    "policy_sources": len(
                        {item["source_file"] for item in policy}
                    ),
                    "sentiment_sources": len(
                        {item["source_file"] for item in sentiment}
                    ),
                },
                "evidence": evidence,
            })

    if MAX_CASES > 0:
        packages = packages[:MAX_CASES]
    return packages


def shuffled_package(package: dict[str, Any], seed: int) -> dict[str, Any]:
    value = json.loads(json.dumps(package, ensure_ascii=False))
    random.Random(seed).shuffle(value["evidence"])
    return value


def valid_ids(package: dict[str, Any], corpus: str | None = None) -> set[str]:
    return {
        item["evidence_id"]
        for item in package["evidence"]
        if corpus is None or item["corpus"] == corpus
    }


def pairwise_jaccard(values: list[set[str]]) -> float:
    """Mean pairwise Jaccard; zero means repeated-run stability is unavailable."""
    if len(values) < 2:
        return 0.0

    scores: list[float] = []
    for left, right in itertools.combinations(values, 2):
        union = left | right
        scores.append(1.0 if not union else len(left & right) / len(union))
    return float(np.mean(scores)) if scores else 0.0


def parse_bool(value: Any) -> bool | None:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None

    normalised = str(value).strip().lower()
    if normalised in {"1", "true", "yes", "y"}:
        return True
    if normalised in {"0", "false", "no", "n"}:
        return False
    return None


def normalise_ids(value: Any) -> list[str]:
    if not isinstance(value, list):
        return []
    return list(dict.fromkeys(
        str(item).strip() for item in value if str(item).strip()
    ))


def normalise_findings(value: Any) -> list[dict[str, Any]]:
    if not isinstance(value, list):
        return []

    findings: list[dict[str, Any]] = []
    for position, item in enumerate(value, start=1):
        if not isinstance(item, dict):
            continue

        record = dict(item)
        record["finding_id"] = str(
            record.get("finding_id") or f"S{position}"
        ).strip()
        record["classification"] = str(
            record.get("classification", "insufficient_evidence")
        ).strip().lower()

        if record["classification"] not in ALLOWED_CLASSIFICATIONS:
            record["classification"] = "insufficient_evidence"

        record["gap_label"] = str(record.get("gap_label", "")).strip()
        record["explanation"] = str(record.get("explanation", "")).strip()

        for key in [
            "policy_evidence_ids",
            "sentiment_evidence_ids",
            "counterevidence_ids",
        ]:
            record[key] = normalise_ids(record.get(key, []))

        try:
            record["confidence"] = min(
                1.0, max(0.0, float(record.get("confidence", 0.0)))
            )
        except Exception:
            record["confidence"] = 0.0

        findings.append(record)

    return findings


def finding_signature(finding: dict[str, Any]) -> str:
    label = re.sub(
        r"\s+",
        " ",
        str(finding.get("gap_label", "")).strip().lower(),
    )
    return f"{finding.get('classification', '')}::{label}"


def json_cell(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False)


packages = build_packages(empirical)
print("Analysis packages:", len(packages))

if packages:
    display(pd.DataFrame([
        {
            "analysis_id": package["analysis_id"],
            "scope": package["scope"],
            **package["evidence_counts"],
        }
        for package in packages
    ]))

In [ ]:
SEMANTIC_ANALYSIS_PROMPT = r"""
You analyse semantic policy--sentiment gaps using only the supplied cleaned
source sentences.

Rules:
- Use no outside knowledge.
- Treat evidence IDs as immutable.
- Do not treat absence from a small evidence batch as proof of corpus-wide absence.
- Do not infer true real-world causality.
- A substantive finding must cite evidence from both policy and sentiment.
- Consider supporting and contradictory evidence.
- Alignment and insufficient evidence are valid outcomes.
- Return at most the requested number of findings.

Return one JSON object with exactly this structure:
{
  "analysis_id": "exact input analysis_id",
  "findings": [
    {
      "finding_id": "S1, S2, ...",
      "classification": "policy_gap | sentiment_gap | partial_alignment | alignment | insufficient_evidence",
      "gap_label": "short descriptive label",
      "explanation": "no more than 90 words",
      "policy_evidence_ids": ["exact policy evidence IDs"],
      "sentiment_evidence_ids": ["exact sentiment evidence IDs"],
      "counterevidence_ids": ["exact evidence IDs"],
      "confidence": 0.0
    }
  ],
  "limitations": ["short limitations"]
}

Definitions:
- policy_gap: a policy emphasis has weak or conflicting sentiment support.
- sentiment_gap: a sentiment concern or expectation has weak or conflicting policy support.
- partial_alignment: the corpora overlap but differ materially in emphasis or framing.
- alignment: the supplied evidence conveys substantially corresponding meaning.
- insufficient_evidence: the supplied evidence cannot justify a reliable comparison.
""".strip()


SEMANTIC_VERIFICATION_PROMPT = r"""
Independently verify the candidate findings using only the reordered evidence.

Check:
- the analysis_id;
- every cited evidence ID;
- support from both corpora;
- omitted counterevidence;
- the classification;
- whether the explanation follows from the cited passages;
- whether the finding improperly treats batch-level absence as corpus-wide absence.

Return one JSON object with exactly this structure:
{
  "analysis_id": "exact input analysis_id",
  "verdict": "accept | revise | reject",
  "verified_findings": [
    {
      "finding_id": "S1, S2, ...",
      "classification": "policy_gap | sentiment_gap | partial_alignment | alignment | insufficient_evidence",
      "gap_label": "short descriptive label",
      "explanation": "no more than 90 words",
      "policy_evidence_ids": ["exact policy evidence IDs"],
      "sentiment_evidence_ids": ["exact sentiment evidence IDs"],
      "counterevidence_ids": ["exact evidence IDs"],
      "confidence": 0.0
    }
  ],
  "invalid_evidence_ids": ["IDs"],
  "unresolved_counterevidence_ids": ["IDs"],
  "evidence_faithfulness": 0.0,
  "reason": "no more than 70 words"
}

Use an empty verified_findings list when the candidate must be rejected.
""".strip()


ANALYSIS_KEYS = {"analysis_id", "findings", "limitations"}
VERIFICATION_KEYS = {
    "analysis_id",
    "verdict",
    "verified_findings",
    "invalid_evidence_ids",
    "unresolved_counterevidence_ids",
    "evidence_faithfulness",
    "reason",
}

In [ ]:
def run_key(analysis_id: str, run_index: int) -> tuple[str, int]:
    return analysis_id, run_index


def load_previous_runs() -> dict[tuple[str, int], dict[str, Any]]:
    previous: dict[tuple[str, int], dict[str, Any]] = {}
    if not RESUME_RUNS or not RUNS_PATH.exists():
        return previous

    with RUNS_PATH.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                key = run_key(str(row["analysis_id"]), int(row["run_index"]))
                previous[key] = row
            except Exception:
                continue
    return previous


def save_runs(rows: Iterable[dict[str, Any]]) -> None:
    with RUNS_PATH.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def validate_verified_findings(
    package: dict[str, Any],
    verification: dict[str, Any],
) -> tuple[list[dict[str, Any]], list[str]]:
    errors: list[str] = []
    findings = normalise_findings(
        verification.get("verified_findings", [])
    )

    all_ids = valid_ids(package)
    policy_ids = valid_ids(package, "policy")
    sentiment_ids = valid_ids(package, "sentiment")

    valid_findings: list[dict[str, Any]] = []
    for finding in findings:
        cited_policy = set(finding["policy_evidence_ids"])
        cited_sentiment = set(finding["sentiment_evidence_ids"])
        cited_counter = set(finding["counterevidence_ids"])

        evidence_is_valid = (
            cited_policy.issubset(policy_ids)
            and cited_sentiment.issubset(sentiment_ids)
            and cited_counter.issubset(all_ids)
        )

        classification = finding["classification"]
        substantive = classification in SUBSTANTIVE_CLASSIFICATIONS
        both_corpora = bool(cited_policy) and bool(cited_sentiment)

        if not evidence_is_valid:
            errors.append(f"{finding['finding_id']}: invalid evidence ID")
            continue
        if substantive and not both_corpora:
            errors.append(
                f"{finding['finding_id']}: substantive finding lacks evidence "
                "from both corpora"
            )
            continue
        if finding["confidence"] < MIN_CONFIDENCE:
            errors.append(f"{finding['finding_id']}: confidence below threshold")
            continue

        valid_findings.append(finding)

    return valid_findings, errors


previous_runs = load_previous_runs()
run_rows: list[dict[str, Any]] = []

for package_index, package in enumerate(packages):
    analysis_id = package["analysis_id"]

    for zero_based_run in range(ANALYSIS_RUNS):
        run_index = zero_based_run + 1
        key = run_key(analysis_id, run_index)

        if key in previous_runs:
            print(f"Reusing {analysis_id}, run {run_index}")
            run_rows.append(previous_runs[key])
            continue

        if RUN_AGENT:
            candidate = ask_agent(
                SEMANTIC_ANALYSIS_PROMPT,
                shuffled_package(
                    package,
                    RANDOM_STATE + package_index * 100 + zero_based_run,
                ),
                action=f"Analysing {analysis_id}, run {run_index}",
                required_keys=ANALYSIS_KEYS,
                temperature=0.0,
            )

            verification = ask_agent(
                SEMANTIC_VERIFICATION_PROMPT,
                {
                    "analysis_id": analysis_id,
                    "candidate": candidate,
                    "case": shuffled_package(
                        package,
                        RANDOM_STATE
                        + 10000
                        + package_index * 100
                        + zero_based_run,
                    ),
                },
                action=f"Verifying {analysis_id}, run {run_index}",
                required_keys=VERIFICATION_KEYS,
                temperature=0.0,
            )
        else:
            candidate = {
                "analysis_id": analysis_id,
                "findings": [],
                "limitations": ["DeepSeek execution is disabled."],
                "status": "not_run",
            }
            verification = {
                "analysis_id": analysis_id,
                "verdict": "reject",
                "verified_findings": [],
                "invalid_evidence_ids": [],
                "unresolved_counterevidence_ids": [],
                "evidence_faithfulness": 0.0,
                "reason": "Set DEEPSEEK_API_KEY and RUN_DEEPSEEK_AGENT=1.",
                "status": "not_run",
            }

        try:
            faithfulness = min(
                1.0,
                max(0.0, float(verification.get("evidence_faithfulness", 0.0))),
            )
        except Exception:
            faithfulness = 0.0

        verdict = str(verification.get("verdict", "reject")).strip().lower()
        analysis_id_matches = (
            str(candidate.get("analysis_id", "")) == analysis_id
            and str(verification.get("analysis_id", "")) == analysis_id
        )

        invalid_evidence_ids = normalise_ids(
            verification.get("invalid_evidence_ids", [])
        )
        unresolved_counterevidence_ids = normalise_ids(
            verification.get("unresolved_counterevidence_ids", [])
        )

        valid_findings, validation_errors = validate_verified_findings(
            package,
            verification,
        )

        machine_accepted = bool(
            analysis_id_matches
            and verdict in {"accept", "revise"}
            and faithfulness >= MIN_FAITHFULNESS
            and not invalid_evidence_ids
            and not unresolved_counterevidence_ids
            and not validation_errors
            and valid_findings
        )

        row = {
            "analysis_id": analysis_id,
            "scope": package["scope"],
            "batch_id": package["batch_id"],
            "run_index": run_index,
            "candidate": candidate,
            "verification": verification,
            "findings": valid_findings,
            "verdict": verdict,
            "evidence_faithfulness": faithfulness,
            "analysis_id_matches": analysis_id_matches,
            "invalid_evidence_ids": invalid_evidence_ids,
            "unresolved_counterevidence_ids": unresolved_counterevidence_ids,
            "validation_errors": validation_errors,
            "machine_accepted": machine_accepted,
            "analysis_model": candidate.get("_agent", {}).get("model"),
            "verification_model": verification.get("_agent", {}).get("model"),
        }

        run_rows.append(row)
        previous_runs[key] = row
        save_runs(previous_runs.values())

print("Completed run records:", len(run_rows))
print("Accepted run records:", sum(row["machine_accepted"] for row in run_rows))

In [ ]:
stability_rows: list[dict[str, Any]] = []
candidate_rows: list[dict[str, Any]] = []

analysis_ids = [package["analysis_id"] for package in packages]

for analysis_id in analysis_ids:
    group = sorted(
        [row for row in run_rows if row["analysis_id"] == analysis_id],
        key=lambda row: row["run_index"],
    )
    if not group:
        continue

    accepted = [row for row in group if row["machine_accepted"]]
    signature_sets = [
        {finding_signature(finding) for finding in row["findings"]}
        for row in accepted
    ]

    stability = pairwise_jaccard(signature_sets)
    repeated_stability_available = len(signature_sets) >= 2

    representative = max(
        accepted,
        key=lambda row: (
            row["evidence_faithfulness"],
            len(row["findings"]),
        ),
        default=None,
    )

    machine_consensus = bool(
        representative
        and len(accepted) >= REQUIRED_ACCEPTED_RUNS
        and repeated_stability_available
        and stability >= MIN_STABILITY
    )

    stability_rows.append({
        "analysis_id": analysis_id,
        "scope": group[0]["scope"],
        "batch_id": group[0]["batch_id"],
        "accepted_runs": len(accepted),
        "required_accepted_runs": REQUIRED_ACCEPTED_RUNS,
        "total_runs": len(group),
        "repeated_stability_available": repeated_stability_available,
        "finding_jaccard_stability": stability,
        "mean_evidence_faithfulness": (
            float(np.mean([
                row["evidence_faithfulness"] for row in accepted
            ]))
            if accepted else 0.0
        ),
        "machine_consensus": machine_consensus,
    })

    if representative:
        for finding in representative["findings"]:
            candidate_rows.append({
                "analysis_id": analysis_id,
                "scope": representative["scope"],
                "batch_id": representative["batch_id"],
                "finding_id": finding["finding_id"],
                "classification": finding["classification"],
                "gap_label": finding["gap_label"],
                "explanation": finding["explanation"],
                "policy_evidence_ids": json_cell(
                    finding["policy_evidence_ids"]
                ),
                "sentiment_evidence_ids": json_cell(
                    finding["sentiment_evidence_ids"]
                ),
                "counterevidence_ids": json_cell(
                    finding["counterevidence_ids"]
                ),
                "confidence": finding["confidence"],
                "evidence_faithfulness": representative[
                    "evidence_faithfulness"
                ],
                "accepted_runs": len(accepted),
                "finding_jaccard_stability": stability,
                "machine_consensus": machine_consensus,
                "machine_retained": machine_consensus,
            })

stability_df = pd.DataFrame(stability_rows)
candidates_df = pd.DataFrame(candidate_rows)

if candidates_df.empty:
    candidates_df = pd.DataFrame(columns=[
        "analysis_id",
        "scope",
        "batch_id",
        "finding_id",
        "classification",
        "gap_label",
        "explanation",
        "policy_evidence_ids",
        "sentiment_evidence_ids",
        "counterevidence_ids",
        "confidence",
        "evidence_faithfulness",
        "accepted_runs",
        "finding_jaccard_stability",
        "machine_consensus",
        "machine_retained",
    ])

findings_df = candidates_df[
    candidates_df["machine_retained"].eq(True)
].copy()

stability_df.to_csv(STABILITY_PATH, index=False)
candidates_df.to_csv(CANDIDATES_PATH, index=False)
findings_df.to_csv(FINDINGS_PATH, index=False)

print("Stability rows:", len(stability_df))
print("Representative candidates:", len(candidates_df))
print("Machine-retained findings:", len(findings_df))

display(stability_df)
display(findings_df.head(20))

In [ ]:
HUMAN_COLUMNS = [
    "human_is_gap",
    "human_category",
    "human_evidence_faithful",
    "human_confirmed",
    "human_notes",
]
KEY_COLUMNS = ["analysis_id", "finding_id"]


def build_review_table(candidates: pd.DataFrame) -> pd.DataFrame:
    review = candidates.copy()

    # Preserve human labels from a previous review file.
    if REVIEW_PATH.exists():
        previous = pd.read_csv(REVIEW_PATH).fillna("")
        available = [
            column for column in HUMAN_COLUMNS
            if column in previous.columns
        ]
        if all(column in previous.columns for column in KEY_COLUMNS):
            previous_labels = previous[
                KEY_COLUMNS + available
            ].drop_duplicates(KEY_COLUMNS)

            review = review.merge(
                previous_labels,
                on=KEY_COLUMNS,
                how="left",
            )

    for column in HUMAN_COLUMNS:
        if column not in review.columns:
            review[column] = ""
        else:
            review[column] = review[column].fillna("")

    preferred_order = [
        "analysis_id",
        "scope",
        "batch_id",
        "finding_id",
        "classification",
        "gap_label",
        "explanation",
        "policy_evidence_ids",
        "sentiment_evidence_ids",
        "counterevidence_ids",
        "confidence",
        "evidence_faithfulness",
        "accepted_runs",
        "finding_jaccard_stability",
        "machine_consensus",
        "machine_retained",
        *HUMAN_COLUMNS,
    ]
    return review[
        [column for column in preferred_order if column in review.columns]
    ]


review_df = build_review_table(candidates_df)
review_df.to_csv(REVIEW_PATH, index=False)

print("Human-review file:", REVIEW_PATH)
print(
    "Complete human_is_gap with yes/no, save the CSV, "
    "then rerun only the next evaluation cell."
)
display(review_df.head(20))

In [ ]:
def evaluate_human_review(review_path: Path) -> pd.DataFrame:
    if not review_path.exists():
        return pd.DataFrame([{
            "labelled_findings": 0,
            "true_positives": 0,
            "false_positives": 0,
            "false_negatives": 0,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "status": "human review file not found",
        }])

    review = pd.read_csv(review_path).fillna("")
    if "human_is_gap" not in review.columns:
        raise ValueError("human_is_gap is missing from the review file.")

    review["human_label"] = review["human_is_gap"].map(parse_bool)
    labelled = review[review["human_label"].notna()].copy()

    if labelled.empty:
        return pd.DataFrame([{
            "labelled_findings": 0,
            "true_positives": 0,
            "false_positives": 0,
            "false_negatives": 0,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "status": "enter yes/no labels in human_is_gap",
        }])

    labelled["machine_label"] = labelled["machine_retained"].map(parse_bool)
    labelled["machine_label"] = labelled["machine_label"].fillna(False).astype(bool)
    labelled["human_label"] = labelled["human_label"].astype(bool)

    tp = int((labelled["machine_label"] & labelled["human_label"]).sum())
    fp = int((labelled["machine_label"] & ~labelled["human_label"]).sum())
    fn = int((~labelled["machine_label"] & labelled["human_label"]).sum())

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return pd.DataFrame([{
        "labelled_findings": len(labelled),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "status": "complete",
    }])


evaluation_df = evaluate_human_review(REVIEW_PATH)
evaluation_df.to_csv(EVALUATION_PATH, index=False)

print("Evaluation file:", EVALUATION_PATH)
display(evaluation_df)

## Output files

The notebook writes:

- `semantic_agent_runs.jsonl`: complete analysis and verification records;
- `semantic_agent_stability.csv`: accepted-run and Jaccard stability summary;
- `semantic_agent_candidates.csv`: representative verified candidates, including unstable ones;
- `semantic_agent_findings.csv`: findings retained after repeated-run consensus;
- `semantic_agent_human_review.csv`: review template with preserved human labels;
- `semantic_agent_evaluation.csv`: precision, recall and \(F_1\) after human labelling.

To force a completely new DeepSeek run, delete `semantic_agent_runs.jsonl` or set:

```bash
export AGENT_RESUME_RUNS=0
```
